In [ ]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# 2. Clone the original YOLOv9 research repository
!git clone https://github.com/WongKinYiu/yolov9.git
%cd yolov9

# 3. Install the required dependencies
!pip install -r requirements.txt -q
print("✅ YOLOv9 Research Environment Ready!")

Cloning into 'yolov9'...
remote: Enumerating objects: 781, done.
remote: Total 781 (delta 0), reused 0 (delta 0), pack-reused 781 (from 1)
Receiving objects: 100% (781/781), 3.27 MiB | 9.73 MiB/s, done.
Resolving deltas: 100% (330/330), done.
/content/yolov9
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.0 MB/s eta 0:00:00
✅ YOLOv9 Research Environment Ready!


In [ ]:
# 1. Re-apply the PyTorch 2.6 security patch to YOLOv9
!sed -i "s/ckpt = torch.load(attempt_download(w), map_location='cpu')/ckpt = torch.load(attempt_download(w), map_location='cpu', weights_only=False)/g" /content/yolov9/models/experimental.py
print("✅ PyTorch patch re-applied!")

# 2. Run the ONNX export
!python export.py \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--include onnx \
--simplify \
--img 640 \
--batch 1

print("\n🚀 Export finished! Check your weights folder for best.onnx")

✅ PyTorch patch re-applied!
export: data=data/coco.yaml, weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], imgsz=[640], batch_size=1, device=cpu, half=False, inplace=False, keras=False, optimize=False, int8=False, dynamic=False, simplify=True, opset=12, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx']
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cpu CPU

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs

PyTorch: starting from /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt with output shape (1, 10, 8400) (390.8 MB)

ONNX: starting export with onnx 1.20.1...
ONNX: export failure ❌ 0.2s: No module named 'onnxscript'

🚀 Export finished! Check your weights folder for best.onnx


In [ ]:
# 1. Install the missing ONNX translation library
!pip install onnxscript -q

# 2. Re-run the export!
!python export.py \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--include onnx \
--simplify \
--img 640 \
--batch 1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.6 MB/s eta 0:00:00
export: data=data/coco.yaml, weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], imgsz=[640], batch_size=1, device=cpu, half=False, inplace=False, keras=False, optimize=False, int8=False, dynamic=False, simplify=True, opset=12, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx']
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cpu CPU

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs

PyTorch: starting from /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt with output shape (1, 10, 8400) (390.8 MB)

ONNX: starting export with onnx 1.20.1...
W0316 15:25:08.442000 3495 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exp

In [ ]:
!pip install onnx
!pip install onnxruntime

In [ ]:
# 1. Apply the PyTorch 2.6 security patch so the .pt file can load
!sed -i "s/ckpt = torch.load(attempt_download(w), map_location='cpu')/ckpt = torch.load(attempt_download(w), map_location='cpu', weights_only=False)/g" /content/yolov9/models/experimental.py

# 2. Navigate to the YOLOv9 folder
%cd /content/yolov9

# 3. Run the heavy .pt model
# --project forces the images to save straight to Google Drive
# 2>&1 | tee forces the terminal output to save to a text file in Drive
!echo -e "\n⏳ [1/2] Testing heavy .pt model and saving everything to Drive..."
!python detect_dual.py \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt \
--source /content/yolov9/mini_test_benchmark \
--device cpu \
--img 640 \
--project /content/drive/MyDrive/VOC_PCB \
--name benchmark_pt_images \
--exist-ok 2>&1 | tee /content/drive/MyDrive/VOC_PCB/benchmark_pt_log.txt

# 4. Run the lightweight .onnx model
!echo -e "\n⚡ [2/2] Testing lightweight .onnx model and saving everything to Drive..."
!python detect.py \
--weights /content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.onnx \
--source /content/yolov9/mini_test_benchmark \
--device cpu \
--img 640 \
--project /content/drive/MyDrive/VOC_PCB \
--name benchmark_onnx_images \
--exist-ok 2>&1 | tee /content/drive/MyDrive/VOC_PCB/benchmark_onnx_log.txt

!echo -e "\n✅ ALL DONE! Check your VOC_PCB folder in Google Drive."

/content/yolov9

⏳ [1/2] Testing heavy .pt model and saving everything to Drive...
detect_dual: weights=['/content/drive/MyDrive/VOC_PCB/yolo_results/yolov9-voc-pcb/weights/best.pt'], source=/content/yolov9/mini_test_benchmark, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=cpu, view_img=False, save_txt=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=/content/drive/MyDrive/VOC_PCB, name=benchmark_pt_images, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLO 🚀 v0.1-104-g5b1ea9a Python-3.12.12 torch-2.10.0+cpu CPU

Fusing layers... 
yolov9-c summary: 604 layers, 50709828 parameters, 0 gradients, 236.7 GFLOPs
image 1/10 /content/yolov9/mini_test_benchmark/l_light_01_missing_hole_01_1_600.jpg: 640x640 1 missing hole, 4951.2ms
image 2/10 /content/yolov9/mini_test_benchmark/l_light_01_missing